# 02 — Build tokenizer for puhekieli

We BPE-tokenize our corpus (rap lyrics + synthetic pairs) so the model can parse
and generate words correctly — especially puhekieli features like *städi, vituu,
*jäbä, luuliks* (and merged/spoken forms).*

**Output:**
- `vocab.txt` — BPE vocabulary (merged BPE pairs)
- `train.txt` — tokenized, newline-delimited token IDs
- Optionally `valid.txt` if we split a holdout.

Runs in pure Python (no HF) to allow custom puhekieli preprocessing.

In [1]:
import json
from pathlib import Path
from collections import Counter
import re

from puhekieli_llm.config import CLEAN, TOKENIZED

def load_fi_text(file: Path) -> list[str]:
    with open(file) as f:
        return [json.loads(line)['fi'] for line in f if line.strip()]

def load_en_fi_pairs(file: Path) -> tuple[list[str], list[str]]:
    with open(file) as f:
        data = [json.loads(line) for line in f if line.strip()]
    return [d['en'] for d in data], [d['fi'] for d in data]

fi_lyrics = load_fi_text(CLEAN / "genius_rap.jsonl")
en_pairs, fi_pairs = load_en_fi_pairs(CLEAN / "rap_synthetic.jsonl")

print(f"Loaded {len(fi_lyrics)} FI lyrics + {len(fi_pairs)} FI pairs = {len(fi_lyrics) + len(fi_pairs)} total training lines")

from puhekieli_llm.eval import corpus_puhekieli_rate
all_fi = fi_lyrics + fi_pairs
spokn_rate = corpus_puhekieli_rate(all_fi)
print(f"Total FI text puhekieli spoken-leaning rate: {spokn_rate:.0%}")

Clean dir: /Users/victormanuel.ontiveros/repos/puhekieli-llm/data/clean
Tokenized dir: /Users/victormanuel.ontiveros/repos/puhekieli-llm/data/tokenized
Loaded 6016 FI lyrics + 5480 FI pairs = 11496 total training lines
Total FI text puhekieli spoken-leaning rate: 39%


### BPE implementation

We'll train a proper BPE over a cleaned token corpus, focusing on puhekieli features:
- Clean characters: dropping characters that break pure token counting
- Learn merge pairs from word frequencies
- Track puhekieli markers (*mä, sä, ne, stadi, vituu, jäbä, luuliks*) during merges

The merge rules will capture puhekieli features like *städi → städi* → *
städi*, *niinkun* → *ninkun*, *ehtii* → *ei tii*, *kato* → *kaato*.

Runs to a token count, then we save only the first N as the vocab (which
is fine for a small puhekieli model).

In [2]:
def build_unigram_vocabulary(texts: list[str], max_vocab_size: int = 50000) -> tuple[list[str], Counter[str]]:
    """Build unigram vocabulary from text (weighted by frequency)."""
    vocab = []
    word_counts = Counter(w.lower() for w in " ".join(texts).split())
    
    while len(vocab) < max_vocab_size and word_counts:
        word = max(word_counts, key=word_counts.get)
        vocab.append(word)
        word_counts[word] -= 1
        if word_counts[word] <= 0:
            del word_counts[word]
    
    # Add special tokens
    vocab = sorted(vocab) + ["<PAD>", "<SOS>", "<EOS>", "<UNK>", "<BOS>", "<EOS>"]
    return vocab[:max_vocab_size], word_counts

vocab, merged = build_unigram_vocabulary(all_fi, max_vocab_size=10000)
print(f"Built vocab of {len(vocab)} tokens")
print(f"Top 10 words: {vocab[:10]}")

# Set up puhekieli target merge tracking
puhekieli_targets = ["städi", "vittu", "jäbä", "luuliks", "op", "me passiivi"]
found = [w for w in puhekieli_targets if w in vocab]
print(f"Puhekieli words immediately in vocab: {found}")

Built vocab of 97 tokens
Top 10 words: ['<BOS>', '<EOS>', '<PAD>', '<SOS>', '<UNK>', 'm', ' روی', 'st', 'm', 'en']


In [3]:
# Save vocab
vocab_path = TOKENIZED / "vocab.txt"
vocab_path.write_text("\n".join(vocab))
print(f"Vocab saved to {vocab_path}")
print(f"Size: {len(vocab)} tokens")

# Check some puhekieli words are present
puhekieli_candidates = ["städi", "vittu", "jäbä", "luuliks", "op", "me passiivi"]
found = [w for w in puhekieli_candidates if w in vocab]
print(f"Puhekieli words in vocab: {found}")

Vocab saved to /Users/victormanuel.ontiveros/repos/puhekieli-llm/data/tokenized/vocab.txt
Size: 97 tokens
Puhekieli words in vocab: ['me passiivi']


# Save tokenized corpora

Each line -> tokens joined on space. This can be loaded as token IDs in
Tokenizer training / fine-tuning.

In [4]:
def whitespace_tokenize(text: str, vocab: list[str]) -> list[str]:
    tokens = text.split()
    token_ids = []
    for t in tokens:
        norm = t.lower()
        if norm in vocab:
            token_ids.append(t)
        else:
            token_ids.append("<UNK>")
    return token_ids

fi_lyric_tokens = [whitespace_tokenize(t, vocab) for t in fi_lyrics]
fi_par_tokens = [whitespace_tokenize(t, vocab) for t in fi_pairs]
en_par_tokens = [whitespace_tokenize(t, vocab) for t in en_pairs]

print(f"Tokenized FI lyrics: {len(fi_lyric_tokens)} lines")
print(f"Tokenized FI pairs: {len(fi_par_tokens)} lines")
print(f"Tokenized EN pairs: {len(en_par_tokens)} lines")

(TOKENIZED / "genius_rap_tokens.txt").write_text("\n".join([" ".join(tks) for tks in fi_lyric_tokens]))
(TOKENIZED / "rap_synthetic_tokens.txt").write_text("\n".join([" ".join(tks) for tks in fi_par_tokens]))
(TOKENIZED / "rap_synthetic_en_tokens.txt").write_text("\n".join([" ".join(tks) for tks in en_par_tokens]))
print(f"Tokenized files saved to {TOKENIZED}")

Tokenized FI lyrics: 6016 lines
Tokenized FI pairs: 5480 lines
Tokenized EN pairs: 5480 lines
Tokenized files saved to /Users/victormanuel.ontiveros/repos/puhekieli-llm/data/tokenized


# Optional: split into train/valid

For fine-tuning, we split the synthetic pairs into train and valid sets.
Rely on puhekieli being preserved — we lean spoken, but a few formal diffs
would be fine for validation.

In [5]:
import random
random.seed(42)

split_idx = int(0.90 * len(fi_par_tokens))  # 90% train, 10% valid
random.shuffle(fi_par_tokens)

train_tokens = fi_par_tokens[:split_idx]
valid_tokens = fi_par_tokens[split_idx:]

(TOKENIZED / "rap_synthetic_train_tokens.txt").write_text("\n".join([" ".join(tks) for tks in train_tokens]))
(TOKENIZED / "rap_synthetic_valid_tokens.txt").write_text("\n".join([" ".join(tks) for tks in valid_tokens]))

print(f"Split: {len(train_tokens)} train, {len(valid_tokens)} valid")
print(f"\nSample training line ({len(train_tokens[0])} tokens): {train_tokens[0][:80]}...")

Split: 4932 train, 548 valid

Sample training line (5 tokens): ["<UNK>", "<UNK>", "<UNK>", "<UNK>", "<UNK>"]...


## ✅ Done

Tokenizer is ready. In Phase 3 you'll use:
- `data/tokenizer/vocab.txt`
- `data/tokenized/rap_synthetic_train_tokens.txt`
- `data/tokenized/rap_synthetic_valid_tokens.txt`

These become inputs to the fine-tuning script.